# Konu 9 — Video, arayüzden

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 8'de görselleri koddan ve toplu olarak ürettin. Bugün görsele **zaman** ekliyoruz: kısa video klipler.
Videoyu **arayüzde** üreteceksin (Google Flow; adım adım anlatımı `ders_notu.md`'de). Görselde kullandığımız Cloudflare'de
video modeli yok; Google'ın video API'si ise ücretli: koddan üretmek her saniye için para ödemek demek. Bu defterin iki işi var:

1. **Maliyet.** Aynı klip ücretli bir API'yle koddan üretilseydi kaça mal olurdu? Süre ve çözünürlük fiyatı nasıl değiştiriyor?
2. **Kayıt.** Arayüzde ürettiğin her klibin yönergesi ve ayarları nerede saklanacak? Senin JSON günlüğünde (Konu 8'deki üretim günlüğü gibi).

Bu defter **anahtar istemez, internete çıkmaz.** Arayüze kişisel Google hesabınla gireceksin (18 yaş ve üstü);
Google'ın kredi sayfasına göre aboneliği olmayan hesaba da her gün bir miktar ücretsiz kredi veriliyor.

## Isınma — Bir klip kaç kare?

Video art arda gelen karelerden oluşur. Sinema filmi saniyede 24 kare gösterir; göz kareleri ayrı ayrı değil, hareket olarak görür.

**Çalıştırmadan önce tahmin et, sohbete yaz:** 8 saniyelik bir klip, saniyede 24 kareyle kaç kare eder?

In [ ]:
kare_hizi = 24
sure = 8
kare = sure * kare_hizi
print(sure, "saniye:", kare, "kare")

192 kare. Video modeli bu kareleri tek tek değil, birbirini **tutacak** biçimde birlikte kurar: bir fincan kare kare şekil
değiştirmemeli, ışık bir anda zıplamamalı. Konu 8'de bir görsel birkaç saniyede geliyordu; 192 tutarlı kareyi kurmak dakikalar
sürebilir. Videonun görselden hem yavaş hem pahalı olmasının nedeni bu.

## Adım 1 — Arayüzde ilk klip: başlat ve bırak

`ders_notu.md`'nin **Adım 1** bölümünü izleyerek Google Flow'u aç ve **metinden** bir klip başlat. Yönerge İngilizce
(Konu 7: modeller çoğunlukla İngilizce metinle eğitildi):

> steam rising from a ceramic coffee cup on a wooden desk, soft morning light, slow camera push in

Ayarlar: **16:9**, çıktı sayısı **1**, model Gemini Omni Flash, **720p**, **8 saniye** (12 kredi). **Generate**'e
bastıktan sonra klip hemen gelmez. Sekmeyi açık bırak ve **Adım 2'ye geç**: arayüz
işi arka planda sürdürüyor. Buna **uzun süren iş** denir: isteği başlatırsın, başka bir işe geçersin, sonuç hazır olunca dönersin.

**Sohbete yaz:** düğmeye bastığında ekranda ne gördün: bir yükleme çubuğu mu, bir yüzde mi, bir mesaj mı, yoksa hiçbir şey mi?

Yönergeyi bir yere kopyala; Adım 6'da günlüğe yazacağız.

## Adım 2 — Bir saniye kaç dolar?

Arayüzde para ödemiyorsun; klip ücretsiz kredinle üretiliyor. Aynı klibi **koddan**, ücretli bir video API'siyle üretseydin
her saniyesi için para öderdin. Google'ın resmi fiyat sayfasında (28 Ekim 2026) saniye fiyatı **çözünürlüğe** göre değişiyor.

Fiyatları bir sözlüğe koyuyoruz: anahtar çözünürlük, değer bir saniyenin yaklaşık fiyatı (dolar). Python ondalık sayıyı
virgülle değil **noktayla** yazar: `0.101`.

In [ ]:
fiyatlar = {
    "360p": 0.034,
    "720p": 0.101,
    "1080p": 0.152,
    "4K": 0.304,
}

sure = 8
fiyat = fiyatlar["720p"]
ucret = sure * fiyat
print(sure, "saniye, 720p:", ucret, "USD")

Çözünürlük, görüntünün piksel boyutudur: 720p 1280 × 720, 1080p 1920 × 1080, 4K 3840 × 2160 piksel.
4K, 1080p'nin **dört katı** piksel: eni de boyu da iki katı.

**Çalıştırmadan önce tahmin et, sohbete yaz:** 4K'nın saniye fiyatı 1080p'ninkinin kaç katı? 4 mü, daha az mı, daha çok mu?

In [ ]:
kat = fiyatlar["4K"] / fiyatlar["1080p"]
print("4K, 1080p'nin", kat, "katı fiyat")

İki katı. Fiyat piksel sayısıyla aynı hızda artmıyor; ama her basamak yine de fiyatı belirgin biçimde artırıyor.
Asıl soru "en yüksek kalite hangisi?" değil, "bu klip **nerede** izlenecek?" Çoğu telefon ekranı 4K değil; 4K'nın fazladan
pikselleri orada görünmez, faturada görünür.

## Adım 3 — 20 saniyelik tanıtım: hangi ayar bedeli yarıya indirir?

Bir kafe 20 saniyelik bir tanıtım videosu istiyor ve "4K olsun" diyor. Model tek seferde birkaç saniyelik bir klip üretir;
20 saniyelik video, birkaç klibin kurguda birleşmesiyle çıkar. Hesap için toplam süre yeter.

In [ ]:
sure = 20
cozunurluk = "4K"
fiyat = fiyatlar[cozunurluk]
ucret = sure * fiyat
print(sure, "saniye,", cozunurluk, ":", ucret, "USD")

Kafenin bütçesi bunun **yarısı**: 3,04 dolar. Önce süreye dokunmadan her çözünürlükte 20 saniyenin bedeline bakalım.
Sözlüğün üstünde dönen bir `for` döngüsü anahtarları sırayla verir (Konu 4'teki `for kelime in harita:` gibi). `round(..., 2)`
noktadan sonra iki basamak bırakır (Konu 3).

In [ ]:
for cozunurluk in fiyatlar:
    fiyat = fiyatlar[cozunurluk]
    ucret = round(sure * fiyat, 2)
    print(cozunurluk, ":", ucret, "USD")

4K'dan 1080p'ye inince süre aynı kalıyor, fiyat yarıya iniyor. Şimdi tersini deneyelim: çözünürlük 4K kalsın, süre kısalsın.
Hangi süreler bütçeye sığar?

In [ ]:
butce = 3.04
sureler = [5, 10, 15, 20]
fiyat = fiyatlar["4K"]
for sure in sureler:
    ucret = round(sure * fiyat, 2)
    if ucret <= butce:
        print(sure, "saniye, 4K:", ucret, "USD, bütçeye sığıyor")
    else:
        print(sure, "saniye, 4K:", ucret, "USD, bütçeyi aşıyor")

Yarı bütçeyle iki yol var: **20 saniye 1080p** ya da **10 saniye 4K**. Hangisini önerirdin? Tanıtım nerede gösterilecek:
telefonda, kafenin ekranında, sinema perdesinde? 10 saniyeye inersen hangi çekimleri atardın? **Sohbete yaz.**

Bu hesap kaç kez denediğini saymıyor. Videoda ilk deneme çoğu zaman istediğin gibi olmaz; üç deneme yaparsan aynı 20 saniyenin
faturası üç katına çıkar. Bu yüzden önce kısa ve düşük çözünürlükte dene, beğendiğini büyüt.

## Adım 4 — Klibe dön: izle ve değerlendir

Arayüze dön: klip hazır mı? Hazırsa izle ve indir (indirme adımları ve dosyanın yeri ders notunda: `videolar/metinden.mp4`).
Klip görünmüyorsa sayfayı yenile. **Generate**'e yeniden basma: yeni bir iş başlar ve yeniden kredi düşer.
Tasarımcı gözüyle izle ve **sohbete yaz:**

- Yönergede "slow camera push in" (kamera yavaşça yaklaşır) yazdık. Kamera bunu yaptı mı?
- Buhar nasıl hareket ediyor? Nerede inandırıcı, nerede "yapay" duruyor?
- Klipte ses var mı? Yönergede ses yoktu; ne duyuyorsun, sen ne isterdin?
- **Generate**'e bastıktan kaç saniye sonra geldi? Kaç kredi harcandı (sağ üstteki profil resmine tıkla)?

## Adım 5 — Görselden klip

Şimdi Konu 8'de ürettiğin görsellerden birini videonun **ilk karesi** yapıyoruz (arayüz adımları ders notunda, **Adım 5**).
Konu 8'in görselleri `08-koddan-toplu-uretim` klasöründe: ızgaran `izgara/` içinde (`suluboya_gece.jpg` gibi, stil_renk
adıyla), tek görsellerin `ciktilar/` içinde, ödevinki `odev8/` içinde. Görselin yoksa bu klasördeki `veri/yedek_fincan.jpg`'yi
kullan.

Yönerge artık sahneyi değil, **ne olacağını** anlatıyor; sahne görselde zaten var. Örnek:

> slow camera push in, gentle natural motion, soft morning light

Ayarlar Adım 1'deki gibi; yalnızca süre **6 saniye** (10 kredi). Görselin kare (1:1), video yatay (16:9).

**Üretmeden önce tahmin et, sohbete yaz:** model kenarları ne yapacak? Görseli kırpacak mı, kenarlara yeni bir şey mi ekleyecek?

**Generate**'e bas; beklerken Adım 6'ya geç ve ilk klibin kaydını yaz. Klip gelince karşılaştır:

- Tahminin tuttu mu: görsel kırpıldı mı, kenarlara yeni bir şey mi eklendi?
- Kompozisyon yerinde kaldı mı? Stil (suluboya, linol, düz vektör...) hareket boyunca korundu mu, yoksa video "fotoğrafa" mı kaydı?
- Hangi görseller harekete uygun: derinliği, boşluğu, hareket edebilecek bir öğesi (buhar, yaprak, su) olanlar mı, düz bir afiş mi?

## Adım 6 — Video günlüğü

Konu 8'de her görselin nasıl üretildiğini bir üretim günlüğüne (`uretim_gunlugu.json`) yazmıştık. Arayüzde de bu
alışkanlığı sürdürüyoruz: aylar sonra biri "şunun bir benzerini yapalım" derse yönerge ve ayarlar elinde olmalı. Arayüzün
geçmişi senin dosyan değil; kredi biter, araç değişir, hesap kapanır.

`input` (Konu 8'de de kullandın) ekranın üstünde bir kutu açar; yazdığını **metin** olarak verir, Enter'a basınca hücre devam
eder. Yönergeyi arayüzden kopyalayıp kutuya yapıştır. Modeli, süreyi, çözünürlüğü ve oranı arayüzde seçtiklerinle aynı yap:
model ayarlardaki adıyla, süre **sayı** (tırnaksız), çözünürlük fiyat sözlüğündeki gibi (`"720p"`), oran arayüzdeki gibi
(`"16:9"`). Başka bir arayüz kullandıysan `"arac"` değerini değiştir.

In [ ]:
yonerge = input("Metinden klibin yönergesi: ")
begendim = input("Beğendin mi? (evet / kısmen / hayır): ")

metin_kaydi = {
    "tur": "metinden",
    "arac": "Google Flow",
    "model": "Gemini Omni Flash",
    "yonerge": yonerge,
    "gorsel": "yok",
    "sure": 8,
    "cozunurluk": "720p",
    "oran": "16:9",
    "dosya": "videolar/metinden.mp4",
    "begendim": begendim,
}
print(metin_kaydi)

Kayıtta tohum yok. Konu 7'den beri günlüğe model, tohum ve yönerge yazıyoruz; ama Flow'un video ayarlarında (oran, çıktı
sayısı, model, süre) tohum yok. Konu 7'de gördüğün gibi arayüzler tohumu çoğu zaman gizler. Bu yüzden aynı yönerge bir daha
aynı klibi vermeyebilir; günlük klibin kendisini değil, nasıl üretildiğini saklar.

Görselden klibin kaydı aynı kalıpta. `"gorsel"` alanına kullandığın görselin yolunu yaz. Yol bu defterin klasöründen başlar:
`veri/yedek_fincan.jpg` bu klasörde. Konu 8'deki bir görselse önce bir üst klasöre çık (`..`, Konu 2'deki `../anahtar.txt`
gibi), ör. `"../08-koddan-toplu-uretim/izgara/suluboya_gece.jpg"`.

In [ ]:
yonerge = input("Görselden klibin yönergesi: ")
begendim = input("Beğendin mi? (evet / kısmen / hayır): ")

gorsel_kaydi = {
    "tur": "gorselden",
    "arac": "Google Flow",
    "model": "Gemini Omni Flash",
    "yonerge": yonerge,
    "gorsel": "veri/yedek_fincan.jpg",
    "sure": 6,
    "cozunurluk": "720p",
    "oran": "16:9",
    "dosya": "videolar/gorselden.mp4",
    "begendim": begendim,
}
print(gorsel_kaydi)

İki kaydı bir listeye koyup dosyaya yazıyoruz. Hücre, Konu 8'deki üretim günlüğü hücresiyle aynı kalıpta:

- `json.dump(kayitlar, dosya, ...)`: önce yazılacak şey, sonra dosya.
- `ensure_ascii=False`: Türkçe harfler olduğu gibi yazılır; `indent=2`: dosya okunur olur.

In [ ]:
import json

kayitlar = [metin_kaydi, gorsel_kaydi]
with open("video_gunlugu.json", "w", encoding="utf-8") as dosya:
    json.dump(kayitlar, dosya, ensure_ascii=False, indent=2)

sayi = len(kayitlar)
print(sayi, "kayıt video_gunlugu.json dosyasına yazıldı")

Son olarak günlüğü dosyadan **geri okuyup** soruyoruz: bu klipler ücretli bir API'yle koddan üretilseydi kaça mal olurdu?

- `open(...)` bu kez `"w"` olmadan: dosyayı yalnızca okumak için açar.
- `json.load(dosya)`, `json.dump`'ın tersi: dosyadaki JSON'u yeniden Python listesine çevirir (Konu 8'in alıştırmasında
  karşılaşmıştın). `okunan`, yazdığımız `kayitlar` listesinin aynısı: iki sözlük.
- Neden `kayitlar`'ı değil de dosyayı kullanıyoruz? Defteri kapatıp açınca değişkenler gider, dosya kalır. Günlüğü yarın
  okuyacak program da dosyayı okur.
- Her kaydın çözünürlüğü fiyat sözlüğünde bir anahtar: önce değişkene alıyoruz, sonra fiyatı onunla buluyoruz.

In [ ]:
with open("video_gunlugu.json", encoding="utf-8") as dosya:
    okunan = json.load(dosya)

toplam = 0
for kayit in okunan:
    cozunurluk = kayit["cozunurluk"]
    fiyat = fiyatlar[cozunurluk]
    ucret = kayit["sure"] * fiyat
    toplam = toplam + ucret

toplam = round(toplam, 2)
print("Bu klipler ücretli bir API'yle:", toplam, "USD")

Kendi süre ve çözünürlüğünü yazdıysan senin sayın farklı olabilir. Az bir para gibi görünüyor, ama bu yalnızca **iki klip**:
her biri tek denemede, tek bir kişi için. 12 kişilik bir sınıfta herkes birkaç deneme yapsa fatura hızla büyür. Bugün sıfır
bütçeyle video yapabiliyoruz, çünkü arayüzün ücretsiz günlük kredisi var. Kredi bitince ya ertesi günü beklersin ya da ödersin.

## Bonus

**1. Ücretsiz kredini planla.** Kredi, Flow'un günlük kotası: Konu 8'deki neuron gibi, her üretim ondan düşer. Google'ın
kredi sayfasına göre Flow, aboneliği olmayan hesaba her gün 50 kredi veriyor; bir klibin kaç kredi tuttuğu süreye ve
çözünürlüğe bağlı. Aynı sayfada 720p bir klibin kredisi (28 Ekim 2026): 4 saniye 7,
6 saniye 10, 8 saniye 12, 10 saniye 15 kredi. Arayüzde gördüğün sayılar farklıysa sözlüğü düzelt.

In [ ]:
krediler = {4: 7, 6: 10, 8: 12, 10: 15}
for sure in krediler:
    kredi = krediler[sure]
    saniye_basi = round(kredi / sure, 2)
    print(sure, "saniye:", kredi, "kredi, saniyesi", saniye_basi, "kredi")

Uzun klibin saniyesi daha ucuz; ama deneme için kısa klip daha az kredi harcar. 50 krediyle kaç tane 4 saniyelik ya da kaç
tane 10 saniyelik klip üretebilirsin? Bir günün kredisini nasıl bölerdin: kaç deneme, kaç son hâl?

**2. Dikey video.** Metinden klibin yönergesiyle arayüzde 9:16 oranında (telefon, hikâye) bir klip daha üret, indir ve
`videolar/dikey.mp4` olarak kaydet. Kompozisyon nasıl yeniden kuruldu: fincan nerede duruyor, ne kırpıldı, ne eklendi? Klibi
günlüğe üçüncü kayıt olarak ekle: Adım 6'nın ilk hücresini kopyala; kopyada sözlüğün adını `dikey_kaydi`, `"oran"`ı `"9:16"`,
`"dosya"`yı `"videolar/dikey.mp4"` yap ve kopyayı çalıştır. Sonra `kayitlar` satırını
`[metin_kaydi, gorsel_kaydi, dikey_kaydi]` yap ve Adım 6'nın son iki hücresini yeniden çalıştır.